# Inference using LSDB-style API

In [ ]:
import lsdb

In [ ]:
from distributed import Client, LocalCluster

N_WORKERS, THREADS_PER_WORKER = 1, 1
cluster = LocalCluster(n_workers=N_WORKERS, threads_per_worker=THREADS_PER_WORKER, processes=True)
client = Client(cluster)
client

## Open a HATS catalog with LSDB

Open a TESS catalog with LSDB and use LSDB's `map_partitions` to apply a function to each row in the catalog. In this example, we're applying a function that removes nested rows that contain NaNs.

In [ ]:
from lsdb import ConeSearch

tess = lsdb.open_catalog(
    "https://data.lsdb.io/hats/tess/tess_lightcurve",
    columns=["ticid", "ra_obj", "dec_obj", "lightcurve"],
)

cone = ConeSearch(ra=180.0, dec=0.0, radius_arcsec=1 * 3600)
tess_cone = tess.search(cone)


def drop_nans(df):
    return df.dropna(subset="lightcurve.sap_flux").dropna(subset="lightcurve")


tess_filtered_nans = tess_cone.map_partitions(drop_nans)

In [ ]:
import pandas as pd

from lsdb_ml.hyrax.client import ml_client
from lsdb_ml.hyrax.worker_plugin import predict

config = {
    "model": {"name": "HyraxLoopback"},
    "data_loader": {"batch_size": 64},
    "infer": {"model_weights_file": ""},
}

# ^ Would be nice to hide this meta manipulation
PRED_META = pd.DataFrame(
    {
        "pred": pd.Series(dtype="float32"),
    }
)
met = tess_filtered_nans._operation.meta.copy(deep=True)
met["pred"] = PRED_META["pred"]


with ml_client(
    catalog=tess_filtered_nans, hyrax_config=config, id_field="ticid"
):  # , num_workers=4, mem_limit="4GB"):
    tess_pred = tess_filtered_nans.map_partitions(predict, meta=met)
    tess_pred.compute()

In [ ]:
client.close()
cluster.close()